# Conditional DETR: whale and boat detection

Fine-tunes `microsoft/conditional-detr-resnet-50` on COCO-format chips. It then
writes test predictions to `results/<run_name>/` for the shared eval, and plots
decoder cross-attention for a few detections.

**Data:** `data_dir` points to fake chips from `whaleboat.synthetic` until the
real HIMB chips land. Switching over means changing `data_dir` and nothing else,
as long as the data follows `DATA.md`.

**Hyperparameters:** all of them live in `CONFIG`, which is saved next to the
predictions so every result can be traced back to its settings. The defaults
are the published Conditional DETR fine-tuning settings.

In [ ]:
CONFIG = {
    'data_dir': 'data/synthetic',  # relative to the repo root; "data" for the real chips
    'checkpoint': 'microsoft/conditional-detr-resnet-50',
    'image_size': 800,  # chips are resized to this; larger makes small whales bigger
    'epochs': 50,
    'batch_size': 2,  # fits a 6 GB GPU at 800px
    'lr': 1e-4,  # transformer + heads
    'lr_backbone': 1e-5,  # pretrained ResNet-50
    'weight_decay': 1e-4,
    'grad_clip': 0.1,
    'flip_augment': True,  # random horizontal/vertical flips; overhead imagery has no "up"
    'eval_every': 5,  # epochs between val mAP checks
    'score_threshold': 0.05,  # keep low scores; mAP needs the full PR curve
    'seed': 635,
    'run_name': 'detr',
}

In [ ]:
import json
import math
import random
import subprocess
import sys
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import AutoImageProcessor, AutoModelForObjectDetection

from whaleboat.eval import evaluate

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA = ROOT / CONFIG['data_dir']
OUT = ROOT / 'results' / CONFIG['run_name']
CKPT = ROOT / 'checkpoints' / CONFIG['run_name']
OUT.mkdir(parents=True, exist_ok=True)
CKPT.mkdir(parents=True, exist_ok=True)

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, torch.cuda.get_device_name() if device.type == 'cuda' else '')

## Data

Splits are created once with `whaleboat.splits` (whole scenes per split) and
reused after that. The model's class ids must run 0..N-1, so COCO category ids
are mapped to labels on the way in and back to category ids on the way out.

In [ ]:
if not (DATA / 'splits' / 'test.json').exists():
    subprocess.run(
        [
            sys.executable,
            '-m',
            'whaleboat.splits',
            str(DATA / 'annotations' / 'all.json'),
            str(DATA / 'splits'),
        ],
        check=True,
    )

splits = {
    name: json.loads((DATA / 'splits' / f'{name}.json').read_text())
    for name in ('train', 'val', 'test')
}
categories = sorted(splits['train']['categories'], key=lambda c: c['id'])
CAT_TO_LABEL = {c['id']: i for i, c in enumerate(categories)}
LABEL_TO_CAT = {i: c for c, i in CAT_TO_LABEL.items()}
id2label = {i: c['name'] for i, c in enumerate(categories)}

processor = AutoImageProcessor.from_pretrained(
    CONFIG['checkpoint'],
    size={'shortest_edge': CONFIG['image_size'], 'longest_edge': CONFIG['image_size']},
)


class CocoChips(Dataset):
    def __init__(self, coco, augment=False):
        self.images = coco['images']
        self.anns = defaultdict(list)
        for a in coco['annotations']:
            self.anns[a['image_id']].append(a)
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        info = self.images[i]
        image = Image.open(DATA / 'images' / info['file_name']).convert('RGB')
        anns = [
            {**a, 'bbox': list(a['bbox']), 'category_id': CAT_TO_LABEL[a['category_id']]}
            for a in self.anns[info['id']]
        ]
        if self.augment:
            w, h = image.size
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                for a in anns:
                    a['bbox'][0] = w - a['bbox'][0] - a['bbox'][2]
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_TOP_BOTTOM)
                for a in anns:
                    a['bbox'][1] = h - a['bbox'][1] - a['bbox'][3]
        enc = processor(
            images=image,
            annotations={'image_id': info['id'], 'annotations': anns},
            return_tensors='pt',
        )
        return enc['pixel_values'][0], enc['labels'][0]


def collate(batch):
    # Zero-pad to the largest image in the batch; pixel_mask marks the real pixels.
    pixel_values, labels = zip(*batch)
    H = max(p.shape[1] for p in pixel_values)
    W = max(p.shape[2] for p in pixel_values)
    padded = torch.zeros(len(pixel_values), 3, H, W)
    mask = torch.zeros(len(pixel_values), H, W, dtype=torch.long)
    for i, p in enumerate(pixel_values):
        padded[i, :, : p.shape[1], : p.shape[2]] = p
        mask[i, : p.shape[1], : p.shape[2]] = 1
    return {'pixel_values': padded, 'pixel_mask': mask, 'labels': list(labels)}


train_loader = DataLoader(
    CocoChips(splits['train'], augment=CONFIG['flip_augment']),
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    collate_fn=collate,
)
val_loader = DataLoader(
    CocoChips(splits['val']), batch_size=CONFIG['batch_size'], collate_fn=collate
)
for name, s in splits.items():
    print(f'{name:>5}: {len(s["images"])} chips, {len(s["annotations"])} objects')

## Model

The COCO-pretrained detection head has 91 classes. It's replaced with a fresh
head for our classes (`ignore_mismatched_sizes`), and everything else is
fine-tuned, with the backbone at a 10x lower learning rate.

In [ ]:
model = AutoModelForObjectDetection.from_pretrained(
    CONFIG['checkpoint'],
    id2label=id2label,
    label2id={v: k for k, v in id2label.items()},
    ignore_mismatched_sizes=True,
).to(device)

backbone = [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad]
rest = [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad]
optimizer = torch.optim.AdamW(
    [{'params': rest, 'lr': CONFIG['lr']}, {'params': backbone, 'lr': CONFIG['lr_backbone']}],
    weight_decay=CONFIG['weight_decay'],
)
print(f'{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters')

In [ ]:
def to_device(batch):
    return {
        'pixel_values': batch['pixel_values'].to(device),
        'pixel_mask': batch['pixel_mask'].to(device),
        'labels': [{k: v.to(device) for k, v in t.items()} for t in batch['labels']],
    }


@torch.no_grad()
def predict(coco, batch_size=4):
    # Returns COCO-results predictions in original chip pixel coordinates.
    model.eval()
    preds = []
    images = coco['images']
    for start in range(0, len(images), batch_size):
        infos = images[start : start + batch_size]
        pil = [Image.open(DATA / 'images' / i['file_name']).convert('RGB') for i in infos]
        enc = processor(images=pil, return_tensors='pt').to(device)
        outputs = model(pixel_values=enc['pixel_values'], pixel_mask=enc['pixel_mask'])
        results = processor.post_process_object_detection(
            outputs,
            threshold=CONFIG['score_threshold'],
            target_sizes=[(i['height'], i['width']) for i in infos],
        )
        for info, r in zip(infos, results):
            for score, label, (x0, y0, x1, y1) in zip(
                r['scores'].tolist(), r['labels'].tolist(), r['boxes'].tolist()
            ):
                preds.append(
                    {
                        'image_id': info['id'],
                        'category_id': LABEL_TO_CAT[label],
                        'bbox': [x0, y0, x1 - x0, y1 - y0],
                        'score': score,
                    }
                )
    return preds

## Train

The loss is the DETR set loss: Hungarian matching between object queries and
ground-truth boxes, then a classification loss plus L1 and GIoU box losses.
Every `eval_every` epochs we score val mAP and keep the best checkpoint by
val AP50. The test split isn't touched until the end.

In [ ]:
history = []
best_ap50 = -1.0
for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    start, train_loss = time.time(), 0.0
    for batch in train_loader:
        loss = model(**to_device(batch)).loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_loss = sum(model(**to_device(b)).loss.item() for b in val_loader) / len(val_loader)
    row = {'epoch': epoch, 'train_loss': train_loss / len(train_loader), 'val_loss': val_loss}

    if epoch % CONFIG['eval_every'] == 0 or epoch == CONFIG['epochs']:
        metrics = evaluate(splits['val'], predict(splits['val']))
        row.update({'val_AP': metrics['AP'], 'val_AP50': metrics['AP50']})
        if metrics['AP50'] > best_ap50:
            best_ap50 = metrics['AP50']
            model.save_pretrained(CKPT)
    history.append(row)
    extra = f'  val AP {row["val_AP"]:.3f}  AP50 {row["val_AP50"]:.3f}' if 'val_AP' in row else ''
    elapsed = time.time() - start
    losses = f'train {row["train_loss"]:.3f}  val {val_loss:.3f}'
    print(f'epoch {epoch:3d}  {losses}{extra}  ({elapsed:.0f}s)')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
epochs = [h['epoch'] for h in history]
ax[0].plot(epochs, [h['train_loss'] for h in history], label='train')
ax[0].plot(epochs, [h['val_loss'] for h in history], label='val')
ax[0].set(xlabel='epoch', ylabel='loss', title='DETR set loss')
ax[0].legend()
evals = [h for h in history if 'val_AP' in h]
ax[1].plot([h['epoch'] for h in evals], [h['val_AP50'] for h in evals], marker='o', label='AP50')
ax[1].plot([h['epoch'] for h in evals], [h['val_AP'] for h in evals], marker='o', label='AP')
ax[1].set(xlabel='epoch', title='val mAP')
ax[1].legend()
plt.tight_layout()
plt.show()

## Test predictions

Reload the best checkpoint (by val AP50), predict on test, and save
predictions, config and training history together. Compare with the CNN using
the shared eval with bootstrap CIs:

```bash
uv run python -m whaleboat.eval data/splits/test.json \
    results/cnn/preds_test.json results/detr/preds_test.json --bootstrap 200
```

In [ ]:
# "eager" attention is slower but returns attention weights, which the cross-attention plots need.
model = AutoModelForObjectDetection.from_pretrained(CKPT, attn_implementation='eager').to(device)
test_preds = predict(splits['test'])
(OUT / 'preds_test.json').write_text(json.dumps(test_preds))
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
(OUT / 'history.json').write_text(json.dumps(history, indent=2))

test_metrics = evaluate(splits['test'], test_preds)
for k in ['AP', 'AP50', 'AP_small', 'AR100'] + [f'AP50_{c["name"]}' for c in categories]:
    print(f'{k:<14}{test_metrics[k]:.3f}')

In [ ]:
def show_predictions(coco, preds, n=6, min_score=0.5):
    # Ground truth in green, predictions above min_score in red.
    by_image = defaultdict(list)
    for p in preds:
        if p['score'] >= min_score:
            by_image[p['image_id']].append(p)
    gt = defaultdict(list)
    for a in coco['annotations']:
        gt[a['image_id']].append(a)
    infos = sorted(coco['images'], key=lambda i: -len(gt[i['id']]))[:n]
    fig, axes = plt.subplots(1, len(infos), figsize=(3.2 * len(infos), 3.4))
    for ax, info in zip(np.atleast_1d(axes), infos):
        ax.imshow(Image.open(DATA / 'images' / info['file_name']))
        for a in gt[info['id']]:
            x, y, w, h = a['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec='lime', lw=1.5))
        for p in by_image[info['id']]:
            x, y, w, h = p['bbox']
            ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, ec='red', lw=1))
            ax.text(
                x,
                y - 3,
                f'{id2label[CAT_TO_LABEL[p["category_id"]]]} {p["score"]:.2f}',
                color='red',
                fontsize=6,
            )
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()


show_predictions(splits['test'], test_preds)

## Cross-attention

Each of the decoder's object queries attends over the encoder's feature map
(one cell per 32×32 pixel patch) to decide what and where its object is. For
the highest-scoring queries on a chip, we average the last decoder layer's
cross-attention over heads and upsample it onto the image. This is the start
of the main analysis: does a whale query look at the body, the wake, the
surrounding glare, or something else?

In [ ]:
@torch.no_grad()
def show_attention(info, top_k=4, layer=-1):
    model.eval()
    image = Image.open(DATA / 'images' / info['file_name']).convert('RGB')
    enc = processor(images=image, return_tensors='pt').to(device)
    out = model(
        pixel_values=enc['pixel_values'], pixel_mask=enc['pixel_mask'], output_attentions=True
    )

    # Conditional DETR scores classes with independent sigmoids, not a softmax.
    scores, labels = out.logits[0].sigmoid().max(-1)
    queries = scores.topk(top_k).indices.tolist()

    _, _, H, W = enc['pixel_values'].shape
    fh, fw = math.ceil(H / 32), math.ceil(W / 32)
    attn = out.cross_attentions[layer][0].mean(0)  # heads averaged -> (queries, fh*fw)
    assert attn.shape[-1] == fh * fw, (attn.shape, fh, fw)

    fig, axes = plt.subplots(1, top_k, figsize=(3.2 * top_k, 3.4))
    for ax, q in zip(axes, queries):
        heat = attn[q].reshape(1, 1, fh, fw)
        heat = F.interpolate(heat, size=(image.height, image.width), mode='bilinear')[0, 0].cpu()
        cx, cy, w, h = out.pred_boxes[0, q].tolist()
        ax.imshow(image)
        ax.imshow(heat, cmap='inferno', alpha=0.55)
        ax.add_patch(
            plt.Rectangle(
                ((cx - w / 2) * image.width, (cy - h / 2) * image.height),
                w * image.width,
                h * image.height,
                fill=False,
                ec='cyan',
                lw=1.2,
            )
        )
        ax.set_title(f'query {q}: {id2label[labels[q].item()]} {scores[q]:.2f}', fontsize=8)
        ax.set_axis_off()
    plt.tight_layout()
    plt.show()


gt_count = defaultdict(int)
for a in splits['test']['annotations']:
    gt_count[a['image_id']] += 1
busiest = max(splits['test']['images'], key=lambda i: gt_count[i['id']])
show_attention(busiest)